# 03 — Training

Trains the GPT model using `configs/run_exp001.json`.  
The device is auto-detected: **CUDA** on cloud GPU, MPS on Apple Silicon, CPU otherwise.

**What to upload to cloud:**
- `src/`
- `configs/run_exp001.json`
- `artifacts/datasets/ds_tinystories_tok_bpe8k_T128/` (the binary shards)
- `artifacts/tokenizers/tok_bpe_8k/`
- `runs/` (empty, with `registry.jsonl`)

**What to download after training:**
- `runs/train/<run_id>/` — checkpoints + resolved config + metrics

In [ ]:
# Cell 1 — project root
# Change this to your cloud path. This is the ONLY line that differs between local and cloud.
import os

# os.chdir("/notebooks/alg3")                        # Nebius / RunPod / cloud
os.chdir("/Users/dmitrit/Documents/dev/alg3")       # local

print("Working directory:", os.getcwd())

In [ ]:
# Cell 2 — install dependencies
# On cloud with NVIDIA GPU, torch is usually pre-installed with CUDA.
# Uncomment the line below if needed.
# !pip install torch tokenizers tensorboard numpy --quiet

In [ ]:
# Cell 3 — device check
import torch

if torch.cuda.is_available():
    device = "cuda"
    print(f"CUDA GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Apple MPS (Metal)")
else:
    device = "cpu"
    print("CPU only")

print(f"Using device: {device}")

In [ ]:
# Cell 4 — patch config for detected device, then run training
# The config JSON has device="mps" by default.
# This cell overrides it in-memory and writes a patched config so the
# training run is fully reproducible without editing the original JSON.
import json, sys, tempfile, pathlib
sys.path.insert(0, "src")

with open("configs/run_exp001.json") as f:
    cfg = json.load(f)

cfg["environment"]["device"] = device

# Write a temporary patched config so resolved_run.json records the real device
patched_path = pathlib.Path("configs/_run_exp001_patched.json")
patched_path.write_text(json.dumps(cfg, indent=2))

print(f"Device in config: {cfg['environment']['device']}")
print(f"dtype  in config: {cfg['environment']['dtype']}")
print(f"max_steps       : {cfg['training']['max_steps']}")
print()

In [ ]:
# Cell 5 — run training
from train import run_training

run_training(str(patched_path))

# Clean up patched config
patched_path.unlink(missing_ok=True)

In [ ]:
# Cell 6 — plot training loss from metrics.jsonl
import json
from pathlib import Path

# Find most recent training run
train_runs = sorted(Path("runs/train").iterdir(), key=lambda p: p.stat().st_mtime)
run_dir    = train_runs[-1]
print(f"Run: {run_dir.name}")

metrics = []
with open(run_dir / "metrics.jsonl") as f:
    for line in f:
        metrics.append(json.loads(line))

train_steps  = [m["step"]  for m in metrics if m["name"] == "Loss/train_step"]
train_losses = [m["value"] for m in metrics if m["name"] == "Loss/train_step"]
val_steps    = [m["step"]  for m in metrics if m["name"] == "Loss/val"]
val_losses   = [m["value"] for m in metrics if m["name"] == "Loss/val"]

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 4))
    plt.plot(train_steps, train_losses, alpha=0.4, label="train (step)")
    plt.plot(val_steps,   val_losses,   linewidth=2, label="val (eval)")
    plt.xlabel("Step"); plt.ylabel("Loss")
    plt.title(f"Training — {run_dir.name}")
    plt.legend(); plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed — showing raw values instead")
    for s, v in zip(val_steps, val_losses):
        print(f"  step {s:>5}: val_loss {v:.4f}")